# Rice leaf disease: EfficientNet-B0, ViT-B/16, and Hybrid CNN–ViT
Selected source: Rahmi21 Kaggle notebook input **Classification of Rice Diseases Using Leaf Images** only. Do not merge the auxiliary **Data Ekstraksi** input or field-collected data.
Target classes: blast, blight, tungro. Dataset size is measured after download; it is not hard-coded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/rice_disease_dataset')
RAW_DIR = DATA_ROOT / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
!git clone https://github.com/rudinewbie123/rice-disease-hybrid-cnn-vit.git
%cd rice-disease-hybrid-cnn-vit
!pip install -e '.[dev]' kaggle

## Kaggle authentication
Create a Kaggle API token from your Kaggle account settings and upload `kaggle.json`. Do not commit this file or paste its contents into notebook output.

In [ ]:
from google.colab import files
import os, shutil
uploaded = files.upload()  # select kaggle.json
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('Upload kaggle.json from your Kaggle account settings.')
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
shutil.copy('kaggle.json', kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
os.remove('kaggle.json')
print('Kaggle API credentials configured locally in this Colab runtime.')

## Resolve the exact dataset slug
The reference notebook lists the input by display title. Verify the dataset owner/slug in Kaggle before downloading; do not guess it. Search results below are metadata only.

In [ ]:
!kaggle datasets list -s 'Classification of Rice Diseases Using Leaf Images'

In [ ]:
# Copy the exact owner/dataset-slug from the Kaggle search result above.
DATASET_SLUG = 'REPLACE_WITH_VERIFIED_OWNER/DATASET_SLUG'
if DATASET_SLUG.startswith('REPLACE_') or '/' not in DATASET_SLUG:
    raise ValueError('Set DATASET_SLUG to the exact Kaggle dataset slug first.')
DOWNLOAD_DIR = RAW_DIR / 'kaggle_download'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip
print('Downloaded files:', [p.name for p in DOWNLOAD_DIR.iterdir()])

## Audit and stratified split
Set `SOURCE_DIR` to the extracted directory that contains the Blast, Blight, and Tungro class folders. The utility records observed counts and exact duplicate handling in `splits/dataset_audit.json`.

In [ ]:
SOURCE_DIR = DOWNLOAD_DIR  # change if Kaggle archive has an additional enclosing directory
!python scripts/prepare_dataset.py --source-dir '{SOURCE_DIR}' --data-root '{DATA_ROOT}' --seed 42
import json
print((DATA_ROOT / 'splits' / 'dataset_audit.json').read_text())

In [ ]:
import yaml
config_path = Path('configs/default.yaml')
config = yaml.safe_load(config_path.read_text())
config['data_root'] = str(DATA_ROOT)
config['output_dir'] = '/content/drive/MyDrive/rice_disease_outputs'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Configured three classes:', config['class_names'])
print('Observed split audit:', json.loads((DATA_ROOT / 'splits' / 'dataset_audit.json').read_text())['split_counts'])

## Train the three models
Run each model independently using the same fixed train/validation/test manifests. Validation macro-F1 selects the checkpoint; do not use the test set for tuning.

In [ ]:
MODEL = 'efficientnet_b0'  # repeat for vit_b16 and hybrid
!python scripts/train.py --config configs/default.yaml --model {MODEL}

In [ ]:
MODEL = 'vit_b16'
!python scripts/train.py --config configs/default.yaml --model {MODEL}

In [ ]:
MODEL = 'hybrid'
!python scripts/train.py --config configs/default.yaml --model {MODEL}

In [ ]:
!python scripts/evaluate.py --checkpoint /content/drive/MyDrive/rice_disease_outputs/hybrid/best.pt --config configs/default.yaml --split test